In [ ]:
#| default_exp app

# App

> Textual TUI application for FileSorter — keyboard-driven file sorting with Johnny Decimal support.

In [ ]:
#| export
from textual.app import App, ComposeResult
from textual.binding import Binding
from textual.containers import Horizontal, Vertical
from textual.screen import ModalScreen
from textual.widgets import Header, Footer, Static, Label, ListView, ListItem
from textual.reactive import reactive
from rich.text import Text
from pathlib import Path
import subprocess, platform, sys

from filesorter.config import AppConfig, load_config
from filesorter.domain import JDDomain, JDCategory, load_domains, category_path
from filesorter.preview import get_preview
from filesorter.sorter import Sorter

In [ ]:
#| export
class FileInfoPanel(Static):
    """Left panel showing current file metadata."""

    def update_file(self, path: Path | None, index: int, total: int):
        if path is None:
            self.update(Text("No more files to sort!", style="bold green"))
            return
        stat = path.stat()
        size = stat.st_size
        if size < 1024: size_str = f"{size} B"
        elif size < 1024**2: size_str = f"{size/1024:.1f} KB"
        else: size_str = f"{size/1024**2:.1f} MB"
        from datetime import datetime
        modified = datetime.fromtimestamp(stat.st_mtime).strftime("%Y-%m-%d %H:%M")
        suffix = path.suffix.lower() or "unknown"
        text = Text()
        text.append(f"File {index + 1} of {total}\n\n", style="bold cyan")
        text.append(f"{path.name}\n", style="bold white")
        text.append(f"{suffix}  \u2022  {size_str}\n", style="dim")
        text.append(f"Modified: {modified}\n", style="dim")
        self.update(text)

In [ ]:
#| export
class PreviewPanel(Static):
    """Right panel showing file content preview."""

    def update_preview(self, path: Path | None):
        if path is None:
            self.update("")
            return
        result = get_preview(path)
        if result.error:
            self.update(Text(f"\u26a0 Preview error: {result.error}", style="red"))
        elif result.is_image:
            self.update(Text("\n".join(result.lines), style="bold yellow"))
        elif not result.lines:
            self.update(Text("(empty file)", style="dim"))
        else:
            self.update("\n".join(result.lines))

In [ ]:
#| export
class DomainSelector(ModalScreen[JDDomain | None]):
    """Modal screen for selecting a Johnny Decimal domain."""

    BINDINGS = [Binding("escape", "cancel", "Cancel")]

    def __init__(self, domains: list[JDDomain]):
        super().__init__()
        self.domains = domains

    def compose(self) -> ComposeResult:
        yield Vertical(
            Label("SELECT DOMAIN", id="modal-title"),
            ListView(*[
                ListItem(Label(f"  {d.domain_id} \u2014 {d.name.split(' ', 1)[1] if ' ' in d.name else d.name}"), id=f"domain-{i}")
                for i, d in enumerate(self.domains)
            ], id="domain-list"),
            Label("  [ESC] Cancel", id="modal-hint"),
            id="modal-container",
        )

    def on_list_view_selected(self, event: ListView.Selected) -> None:
        idx = int(event.item.id.split("-")[1])
        self.dismiss(self.domains[idx])

    def action_cancel(self) -> None:
        self.dismiss(None)

In [ ]:
#| export
class CategorySelector(ModalScreen[JDCategory | None]):
    """Modal screen for selecting a category within a domain. Type digits to filter by category ID."""

    BINDINGS = [Binding("escape", "cancel", "Cancel")]

    def __init__(self, domain: JDDomain):
        super().__init__()
        self.domain = domain
        self._typed = ""

    def compose(self) -> ComposeResult:
        items = []
        for area in self.domain.areas:
            items.append(ListItem(Label(f"\u2500\u2500 {area.id} {area.name} \u2500\u2500", classes="area-header"), disabled=True))
            if not area.categories:
                items.append(ListItem(Label("  (no categories)", classes="area-header"), disabled=True))
            for cat in area.categories:
                items.append(ListItem(Label(f"  {cat.id} {cat.name}"), id=f"cat-{cat.id}"))
        yield Vertical(
            Label(f"SELECT CATEGORY \u2014 {self.domain.name}", id="modal-title"),
            Label("Type category number or browse:", id="type-hint"),
            ListView(*items, id="category-list"),
            Label("  [ESC] Cancel", id="modal-hint"),
            id="modal-container",
        )

    def on_list_view_selected(self, event: ListView.Selected) -> None:
        if event.item.id and event.item.id.startswith("cat-"):
            cat_id = event.item.id[4:]
            cat = self.domain.get_category(cat_id)
            if cat: self.dismiss(cat)

    def on_key(self, event) -> None:
        if event.key.isdigit():
            self._typed += event.key
            # Check for exact match
            cat = self.domain.get_category(self._typed)
            if cat:
                self.dismiss(cat)
                return
            # Check if any category starts with typed digits
            possible = [c for c in self.domain.all_categories if c.id.startswith(self._typed)]
            if not possible:
                self._typed = ""  # Reset if no match possible
        elif event.key == "backspace":
            self._typed = self._typed[:-1]

    def action_cancel(self) -> None:
        self.dismiss(None)

In [ ]:
#| export
APP_CSS = """
Screen {
    background: $surface;
}

#main-panels {
    height: 1fr;
}

#file-info {
    width: 1fr;
    padding: 1 2;
    border-right: solid $primary;
}

#preview {
    width: 2fr;
    padding: 1 2;
    overflow-y: auto;
}

#status-bar {
    dock: bottom;
    height: 3;
    padding: 0 2;
    background: $boost;
}

#modal-container {
    width: 70;
    height: 80%;
    border: thick $primary;
    background: $surface;
    padding: 1 2;
}

#modal-title {
    text-style: bold;
    color: $accent;
    padding: 0 0 1 0;
}

#modal-hint {
    color: $text-muted;
    padding: 1 0 0 0;
}

#type-hint {
    color: $warning;
}

.area-header {
    color: $accent;
    text-style: bold;
}

DomainSelector, CategorySelector {
    align: center middle;
}
"""

In [ ]:
#| export
class FileSorterApp(App):
    """Keyboard-driven TUI for sorting files into a Johnny Decimal structure."""

    CSS = APP_CSS
    TITLE = "FileSorter"
    BINDINGS = [
        Binding("r", "to_read", "To Read"),
        Binding("f", "filesystem", "FileSystem"),
        Binding("b", "braincave", "Braincave"),
        Binding("s", "skip", "Skip"),
        Binding("u", "undo", "Undo"),
        Binding("d", "delete", "Delete"),
        Binding("o", "open_image", "Open Image"),
        Binding("q", "quit_app", "Quit"),
    ]

    def __init__(self, source_dir: Path | None = None, config_path: Path | None = None):
        super().__init__()
        self.config = load_config(config_path, source_dir=source_dir)
        self.domains = load_domains(self.config.domain_config_dir)
        self.sorter = Sorter()
        self.files: list[Path] = []
        self.current_index = 0
        self._current_preview = None

    def compose(self) -> ComposeResult:
        yield Header()
        with Horizontal(id="main-panels"):
            yield FileInfoPanel(id="file-info")
            yield PreviewPanel(id="preview")
        yield Static(id="status-bar")
        yield Footer()

    def on_mount(self) -> None:
        self.config.ensure_dirs()
        self._refresh_files()
        self._show_current()

    def _refresh_files(self) -> None:
        self.files = self.sorter.get_files(self.config.source_dir, self.config.sorted_dir)

    @property
    def current_file(self) -> Path | None:
        if 0 <= self.current_index < len(self.files): return self.files[self.current_index]
        return None

    def _show_current(self) -> None:
        f = self.current_file
        self.query_one("#file-info", FileInfoPanel).update_file(f, self.current_index, len(self.files))
        self.query_one("#preview", PreviewPanel).update_preview(f)
        self._current_preview = get_preview(f) if f else None
        if f is None:
            self._set_status("All files processed! " + self.sorter.stats.summary().replace("\n", " | "))
        else:
            self.sub_title = f"{self.config.source_dir}"

    def _set_status(self, msg: str) -> None:
        self.query_one("#status-bar", Static).update(Text(msg))

    def _advance(self) -> None:
        self._refresh_files()
        if self.current_index >= len(self.files):
            self.current_index = max(0, len(self.files) - 1)
        self._show_current()

    def action_to_read(self) -> None:
        f = self.current_file
        if f is None: return
        dest = self.sorter.move_file(f, self.config.to_read_dir)
        self._set_status(f"\u2713 Moved \u2192 to-read/{dest.name}")
        self._advance()

    def action_delete(self) -> None:
        f = self.current_file
        if f is None: return
        self.sorter.delete_file(f)
        self._set_status(f"\u2717 Deleted {f.name}")
        self._advance()

    def action_skip(self) -> None:
        if self.current_file is None: return
        self.sorter.skip()
        self.current_index += 1
        if self.current_index >= len(self.files):
            self._refresh_files()
            self.current_index = max(0, len(self.files) - 1)
        self._show_current()
        self._set_status("\u23ed Skipped")

    def action_undo(self) -> None:
        if not self.sorter.can_undo():
            self._set_status("\u26a0 Nothing to undo")
            return
        record = self.sorter.undo()
        if record:
            self._set_status(f"\u21a9 Undone: {record.source.name} restored")
            self._refresh_files()
            # Try to navigate to the restored file
            try:
                self.current_index = self.files.index(record.source)
            except ValueError:
                self.current_index = min(self.current_index, max(0, len(self.files) - 1))
            self._show_current()

    def action_open_image(self) -> None:
        f = self.current_file
        if f is None or self._current_preview is None or not self._current_preview.is_image:
            self._set_status("\u26a0 Not an image file")
            return
        try:
            if platform.system() == "Linux" and "microsoft" in platform.release().lower():
                win_path = subprocess.check_output(["wslpath", "-w", str(f)], text=True).strip()
                subprocess.Popen(["cmd.exe", "/C", "start", "", win_path], cwd="/mnt/c")
            elif platform.system() == "Darwin":
                subprocess.Popen(["open", str(f)])
            else:
                subprocess.Popen(["xdg-open", str(f)])
            self._set_status(f"\ud83d\udcf7 Opened {f.name}")
        except Exception as e:
            self._set_status(f"\u26a0 Could not open: {e}")

    def action_filesystem(self) -> None:
        if self.current_file is None: return
        self._target_root = self.config.filesystem_dir
        self._target_label = "filesystem"
        self.push_screen(DomainSelector(self.domains), self._on_domain_selected)

    def action_braincave(self) -> None:
        if self.current_file is None: return
        self._target_root = self.config.braincave_dir
        self._target_label = "braincave"
        self.push_screen(DomainSelector(self.domains), self._on_domain_selected)

    def _on_domain_selected(self, domain: JDDomain | None) -> None:
        if domain is None:
            self._set_status("Cancelled")
            return
        self.push_screen(CategorySelector(domain), lambda cat: self._on_category_selected(domain, cat))

    def _on_category_selected(self, domain: JDDomain, category: JDCategory | None) -> None:
        if category is None:
            self._set_status("Cancelled")
            return
        f = self.current_file
        if f is None: return
        cat_path = category_path(domain, category)
        dest_dir = self._target_root / cat_path
        dest = self.sorter.move_file(f, dest_dir, domain_id=domain.domain_id)
        self._set_status(f"\u2713 Moved \u2192 {self._target_label}/{cat_path}/{dest.name}")
        self._advance()

    def action_quit_app(self) -> None:
        summary = self.sorter.stats.summary()
        self.exit(message=summary if self.sorter.stats.total > 0 else None)

In [ ]:
#| export
def main():
    """CLI entry point for FileSorter."""
    import argparse
    parser = argparse.ArgumentParser(description="FileSorter \u2014 keyboard-driven file sorter")
    parser.add_argument("--source", type=Path, default=None, help="Source directory (default: ~/Downloads)")
    parser.add_argument("--config", type=Path, default=None, help="Path to config.yaml")
    args = parser.parse_args()
    app = FileSorterApp(source_dir=args.source, config_path=args.config)
    result = app.run()
    if result: print(result)

if __name__ == "__main__":
    main()

## Usage

Run the app:

```bash
uv run python -m filesorter.app
uv run python -m filesorter.app --source ~/Downloads
```

## Tests

The TUI app is tested manually. See PRD for validation criteria.
Automated test: verify app instantiation without crash.

In [ ]:
# Test that app can be instantiated without error
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    source = tmp / "downloads"
    source.mkdir()
    config_dir = Path("/home/jelle/code/hfm-tools/config")

    app = FileSorterApp(source_dir=source)
    # Override domain_config_dir for test
    app.config.domain_config_dir = config_dir
    app.domains = load_domains(config_dir)

    assert app.config.source_dir == source
    assert len(app.domains) == 3
    print("\u2713 App instantiation test passed")